# Siamese BiLSTM — Contrastive Retrieval

Neculoiu et al. (2016) Siamese Recurrent Network for learning text similarity.
Uses word2vec embedding, 4 BiLSTM layers, contrastive loss, Adam optimizer.
Trains on (question, answer) pairs with 4:1 negative ratio.

In [ ]:
!git clone https://github.com/hyperformancelabs/vnlegal-rag.git
%cd vnlegal-rag
!cat data/word2vec/word2vec_part_* > data/word2vec/merged.zip
!7z x data/word2vec/merged.zip -odata/word2vec/ -y

In [21]:
!git fetch origin
!git reset --hard origin/$(git branch --show-current)
!git clean -fd

In [2]:
%cd vnlegal-rag

[Errno 2] No such file or directory: 'vnlegal-rag'
/content/vnlegal-rag


In [3]:
import json, random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

from src.tokenizer import build_vocab, build_embedding_matrix, encode_text, simple_tokenize, PAD_TOKEN, UNK_TOKEN
from src.models.siamese_bilstm import SiameseBiLSTM, ContrastiveLoss

In [4]:
# Device and seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
# Data and artifact paths
DATA_DIR = Path('data/data_ready')
ARTIFACT_DIR = Path('experiments/siamese_1024_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
W2V_PATH = Path('data/word2vec/word2vec_vi_syllables_300dims.txt')
print('DATA_DIR =', DATA_DIR)
print('ARTIFACT_DIR =', ARTIFACT_DIR)

DATA_DIR = data/data_ready
ARTIFACT_DIR = experiments/siamese_bilstm_artifacts


In [6]:
# Load QA splits + corpus
qa_train = pd.read_csv(DATA_DIR / 'qa_train.csv', sep='	')
qa_val   = pd.read_csv(DATA_DIR / 'qa_val.csv',   sep='	')
qa_test  = pd.read_csv(DATA_DIR / 'qa_test.csv',  sep='	')
corpus   = pd.read_csv(DATA_DIR / 'corpus_full.csv', sep='	')

print('train:', qa_train.shape, 'val:', qa_val.shape, 'test:', qa_test.shape)
print('corpus:', corpus.shape)

train: (23408, 11) val: (2902, 11) test: (2832, 11)
corpus: (9582, 5)


In [7]:
# ── Siamese BiLSTM hyperparams (Neculoiu et al. 2016) ──
MAX_LEN = 1024      # questions p95=90, answers p95=370, articles p95=740 — balance

# Architecture (1 layer — 2+ layers collapse signal variance for word-level input)
NUM_BILSTM_LAYERS = 1
HIDDEN_DIM  = 64       # Per direction; output = 128-d after concat
DENSE_DIM   = 128      # Final projection after mean-pool

# Regularization
DROPOUT_RECURRENT = 0.2
DROPOUT_INTER     = 0.4
DROPOUT_OUT       = 0.4

# Contrastive loss (adapted for word-level — margin pushes cos DOWN, not up)
CONTRASTIVE_MARGIN  = 0.5

NEGATIVE_RATIO      = 4
POSITIVE_LOSS_SCALE = 3.0   # scaled up to balance L+/L- for word-level (paper: 0.25)

# Training
BATCH_SIZE = 512
ADAM_LR    = 0.001
GRAD_CLIP  = 5.0
EPOCHS     = 30
PATIENCE   = 5

In [8]:
# Build vocab from all texts (QA + corpus)
all_texts = pd.concat([
    qa_train['question'], qa_val['question'], qa_test['question'],
    qa_train['answer'],   qa_val['answer'],   qa_test['answer'],
    corpus['article_content'],
], ignore_index=True).tolist()

stoi = build_vocab(all_texts, max_vocab=100_000, min_freq=1)
pad_idx = stoi[PAD_TOKEN]
print(f'Vocab size: {len(stoi)}')

Vocab size: 6761


In [9]:
# Build pretrained embedding matrix
embedding_weight, hits = build_embedding_matrix(stoi, embed_dim=300, w2v_path=W2V_PATH)
print(f'Embedding shape: {embedding_weight.shape} | w2v hits: {hits}/{len(stoi) - 2}')
embedding_weight = torch.tensor(embedding_weight, dtype=torch.float32)

Embedding shape: torch.Size([6761, 300]) | w2v hits: 4027/6759


/tmp/ipykernel_35576/1362131438.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  embedding_weight = torch.tensor(embedding_weight, dtype=torch.float32)


In [10]:
# Tokenize questions and answers
train_q_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_train['question']])
train_a_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_train['answer']])
val_q_ids   = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_val['question']])
val_a_ids   = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_val['answer']])
test_q_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_test['question']])
test_a_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in qa_test['answer']])

# Corpus for retrieval eval
corpus_ids  = np.array([encode_text(t, stoi, MAX_LEN) for t in corpus['article_content']])
corpus_test_df = pd.read_csv(DATA_DIR / 'corpus_test.csv', sep='	')
corpus_test_ids = np.array([encode_text(t, stoi, MAX_LEN) for t in corpus_test_df['article_content']])

# Token length stats
q_lens = [len(simple_tokenize(str(t))) for t in qa_train['question']]
a_lens = [len(simple_tokenize(str(t))) for t in qa_train['answer']]
c_lens = [len(simple_tokenize(str(t))) for t in corpus['article_content']]
print(f'Question: max={max(q_lens)} p95={np.percentile(q_lens, 95):.0f}')
print(f'Answer:   max={max(a_lens)} p95={np.percentile(a_lens, 95):.0f}')
print(f'Corpus:   max={max(c_lens)} p95={np.percentile(c_lens, 95):.0f}')
pct = lambda L: 100 * sum(1 for x in L if x > MAX_LEN) / max(len(L), 1)
print(f'Truncated at {MAX_LEN}: q={pct(q_lens):.1f}% a={pct(a_lens):.1f}% c={pct(c_lens):.1f}%')

Question: max=209 p95=90
Answer:   max=989 p95=368
Corpus:   max=92290 p95=743
Truncated at 1024: q=0.0% a=0.0% c=2.8%


In [11]:
# Hard-negative pair dataset via TF-IDF nearest neighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Fit TF-IDF + NN index (memory-efficient, no full similarity matrix)
tfidf = TfidfVectorizer(
    tokenizer=simple_tokenize, max_features=50_000, ngram_range=(1, 2), sublinear_tf=True,
    token_pattern=None,
)
tfidf.fit(qa_train['question'].tolist())

class SiamesePairDataset(Dataset):
    """Pairs with hard negative mining: negatives are most-similar wrong answers."""

    def __init__(self, q_ids, a_ids, q_raw, neg_ratio=4, hard_neg_k=10, hard_ratio=0.5):
        self.q_ids, self.a_ids = q_ids, a_ids
        self.n = len(q_ids)
        self.neg_ratio = neg_ratio
        self.hard_neg_k = hard_neg_k
        self.hard_per_sample = max(1, round(neg_ratio * hard_ratio))
        self.random_per_sample = neg_ratio - self.hard_per_sample

        # Precompute top-k nearest neighbors (excludes self)
        q_vecs = tfidf.transform(q_raw)
        nn = NearestNeighbors(n_neighbors=hard_neg_k + 1, metric='cosine', n_jobs=-1)
        nn.fit(q_vecs)
        _, hard_candidates = nn.kneighbors(q_vecs)
        # Remove self (always first neighbor at distance 0)
        self.hard_candidates = hard_candidates[:, 1:]

    def __len__(self):
        return self.n * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        if idx < self.n:
            i = idx
            label = 1
            return (torch.tensor(self.q_ids[i]), torch.tensor(self.a_ids[i]),
                    torch.tensor(label, dtype=torch.float32))
        else:
            offset = idx - self.n
            bucket = offset // self.neg_ratio
            which = offset % self.neg_ratio
            i = bucket % self.n
            if which < self.hard_per_sample:
                # Hard negative: nearest wrong answer
                j = int(self.hard_candidates[i, which % self.hard_neg_k])
            else:
                # Random negative
                j = random.randrange(self.n)
                while j == i:
                    j = random.randrange(self.n)
            return (torch.tensor(self.q_ids[i]), torch.tensor(self.a_ids[j]),
                    torch.tensor(0, dtype=torch.float32))

In [12]:
# DataLoaders
import sys
nw = 0 if sys.platform == 'win32' else 2
pin = torch.cuda.is_available()

HARD_NEG_K = 10         # pool size for hard negatives
HARD_NEG_RATIO = 0.5    # fraction of negatives that are hard (vs random)

train_raw = qa_train['question'].tolist()
val_raw   = qa_val['question'].tolist()
test_raw  = qa_test['question'].tolist()

train_ds = SiamesePairDataset(
    train_q_ids, train_a_ids, train_raw,
    neg_ratio=NEGATIVE_RATIO, hard_neg_k=HARD_NEG_K, hard_ratio=HARD_NEG_RATIO,
)
val_ds = SiamesePairDataset(
    val_q_ids, val_a_ids, val_raw,
    neg_ratio=NEGATIVE_RATIO, hard_neg_k=HARD_NEG_K, hard_ratio=HARD_NEG_RATIO,
)
test_ds = SiamesePairDataset(
    test_q_ids, test_a_ids, test_raw,
    neg_ratio=NEGATIVE_RATIO, hard_neg_k=HARD_NEG_K, hard_ratio=HARD_NEG_RATIO,
)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=nw, pin_memory=pin)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=nw, pin_memory=pin)

print(f'Train: {len(train_ds):,} pairs ({train_ds.n:,} pos + {train_ds.n * NEGATIVE_RATIO:,} neg)')
print(f'  Hard/random split: {train_ds.hard_per_sample}/{train_ds.random_per_sample}')
print(f'Val:   {len(val_ds):,} pairs')
print(f'Test:  {len(test_ds):,} pairs')

Train: 117,040 pairs (23,408 pos + 93,632 neg)
  Hard/random split: 2/2
Val:   14,510 pairs
Test:  14,160 pairs


In [13]:
# Build Siamese BiLSTM with pretrained word2vec embedding
model = SiameseBiLSTM(
    vocab_size=len(stoi),
    embedding_weight=embedding_weight,
    freeze_embedding=False,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_BILSTM_LAYERS,
    dense_dim=DENSE_DIM,
    dropout_recurrent=DROPOUT_RECURRENT,
    dropout_inter=DROPOUT_INTER,
    dropout_out=DROPOUT_OUT,
    pad_idx=pad_idx,
)
model = model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Params: {total_params:,}')

Params: 2,232,204


In [14]:
# Contrastive loss + Adam (paper §3.1, §4)
criterion = ContrastiveLoss(margin=CONTRASTIVE_MARGIN, positive_scale=POSITIVE_LOSS_SCALE)
optimizer = torch.optim.Adam(model.parameters(), lr=ADAM_LR)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

In [15]:
# Training loop
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for q, a, y in loader:
        q, a, y = q.to(device), a.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            cos = model(q, a)
            loss = criterion(cos, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * q.size(0)
    return total_loss / len(loader.dataset)

In [16]:
# Pair evaluation
def evaluate_pairs(model, loader):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    with torch.no_grad():
        for q, a, y in loader:
            q, a, y = q.to(device), a.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                cos = model(q, a)
                loss = criterion(cos, y)
            total_loss += loss.item() * q.size(0)
            pred = (cos >= CONTRASTIVE_MARGIN).float()
            y_true.extend(y.cpu().tolist())
            y_pred.extend(pred.cpu().tolist())
    acc = sum(1 for t, p in zip(y_true, y_pred) if t == p) / max(len(y_true), 1)
    pos_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1) / max(sum(y_true), 1)
    neg_acc = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0) / max(len(y_true) - sum(y_true), 1)
    return {'loss': total_loss / len(loader.dataset), 'acc': acc,
            'pos_acc': pos_acc, 'neg_acc': neg_acc}

In [17]:
# Retrieval evaluation: encode all → rank → MRR/mAP/recall@k
def retrieval_eval(model, q_ids, a_ids, corpus_ids):
    model.eval()
    bs = 256
    with torch.no_grad():
        q_embs = torch.cat([model.encoder(torch.tensor(q_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(q_ids), bs)], dim=0)
        c_embs = torch.cat([model.encoder(torch.tensor(corpus_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(corpus_ids), bs)], dim=0)
        a_embs = torch.cat([model.encoder(torch.tensor(a_ids[i:i+bs], device=device)).cpu()
                            for i in range(0, len(a_ids), bs)], dim=0)
    sim = F.normalize(q_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T
    a_sim = F.normalize(a_embs, p=2, dim=1) @ F.normalize(c_embs, p=2, dim=1).T
    gt = a_sim.argmax(dim=1).numpy()
    ranks = np.array([(sim[i] > sim[i, gt[i]]).sum() + 1 for i in range(len(q_ids))])
    return {
        'mrr': (1.0 / ranks).mean(),
        'map': (1.0 / ranks).mean(),
        **{f'recall@{k}': (ranks <= k).mean() for k in [1, 5, 10, 20]},
        'mean_rank': ranks.mean(), 'median_rank': np.median(ranks),
    }

In [18]:
# Train with early stopping on val loss
best_path = ARTIFACT_DIR / 'siamese_bilstm_best.pt'
best_val_loss = float('inf')
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val = evaluate_pairs(model, val_loader)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val['loss'],
                    'val_acc': val['acc'], 'val_pos_acc': val['pos_acc'], 'val_neg_acc': val['neg_acc']})

    print(f'Epoch {epoch:2d} | train_loss: {train_loss:.4f} | val_loss: {val["loss"]:.4f} | '
          f'acc: {val["acc"]:.4f} | pos: {val["pos_acc"]:.4f} | neg: {val["neg_acc"]:.4f}')

    if val['loss'] < best_val_loss:
        best_val_loss = val['loss']
        patience_counter = 0
        torch.save(model.state_dict(), best_path)
        print(f'  ✓ saved (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

Epoch  1 | train_loss: 0.0728 | val_loss: 0.0647 | acc: 0.4327 | pos: 0.9976 | neg: 0.2914
  ✓ saved (val_loss=0.0647)
Epoch  2 | train_loss: 0.0513 | val_loss: 0.0508 | acc: 0.4878 | pos: 0.9972 | neg: 0.3604
  ✓ saved (val_loss=0.0508)
Epoch  3 | train_loss: 0.0449 | val_loss: 0.0436 | acc: 0.5484 | pos: 0.9955 | neg: 0.4367
  ✓ saved (val_loss=0.0436)
Epoch  4 | train_loss: 0.0411 | val_loss: 0.0405 | acc: 0.5509 | pos: 0.9966 | neg: 0.4394
  ✓ saved (val_loss=0.0405)
Epoch  5 | train_loss: 0.0374 | val_loss: 0.0374 | acc: 0.5810 | pos: 0.9962 | neg: 0.4772
  ✓ saved (val_loss=0.0374)
Epoch  6 | train_loss: 0.0348 | val_loss: 0.0349 | acc: 0.6111 | pos: 0.9959 | neg: 0.5149
  ✓ saved (val_loss=0.0349)
Epoch  7 | train_loss: 0.0322 | val_loss: 0.0329 | acc: 0.6507 | pos: 0.9890 | neg: 0.5662
  ✓ saved (val_loss=0.0329)
Epoch  8 | train_loss: 0.0304 | val_loss: 0.0330 | acc: 0.6396 | pos: 0.9928 | neg: 0.5513
Epoch  9 | train_loss: 0.0289 | val_loss: 0.0321 | acc: 0.6681 | pos: 0.9910

In [19]:
# Test set pair evaluation
model.load_state_dict(torch.load(best_path, map_location=device))
test = evaluate_pairs(model, test_loader)
print('--- Test Pair Results ---')
print(f'Accuracy:  {test["acc"]:.4f}')
print(f'Pos Acc:   {test["pos_acc"]:.4f}')
print(f'Neg Acc:   {test["neg_acc"]:.4f}')

--- Test Pair Results ---
Accuracy:  0.6814
Pos Acc:   0.9802
Neg Acc:   0.6067


In [20]:
# Full retrieval evaluation
ret = retrieval_eval(model, test_q_ids, test_a_ids, corpus_test_ids)
print('--- Retrieval Results ---')
for k in ['mrr', 'map']:
    print(f'{k.upper():>11}: {ret[k]:.4f}')
for k in [1, 5, 10, 20]:
    print(f'  Recall@{k:>2}: {ret[f"recall@{k}"]:.4f}')
print(f'  Mean rank: {ret["mean_rank"]:.0f}')

--- Retrieval Results ---
        MRR: 0.3563
        MAP: 0.3563
  Recall@ 1: 0.2292
  Recall@ 5: 0.4979
  Recall@10: 0.6084
  Recall@20: 0.7267
  Mean rank: 33


In [21]:
# Save artifacts
torch.save(model.encoder.state_dict(), ARTIFACT_DIR / 'encoder.pt')
torch.save(stoi, ARTIFACT_DIR / 'stoi.pt')

metadata = {
    'model': 'SiameseBiLSTM', 'paper': 'Neculoiu et al. 2016',
    'max_len': MAX_LEN, 'vocab_size': len(stoi), 'embed_dim': 300,
    'num_layers': NUM_BILSTM_LAYERS, 'hidden_dim': HIDDEN_DIM, 'dense_dim': DENSE_DIM,
    'dropout_recurrent': DROPOUT_RECURRENT, 'dropout_inter': DROPOUT_INTER,
    'dropout_out': DROPOUT_OUT, 'margin': CONTRASTIVE_MARGIN,
    'batch_size': BATCH_SIZE, 'adam_lr': ADAM_LR, 'grad_clip': GRAD_CLIP,
    'best_val_loss': best_val_loss, 'test_pair_acc': test['acc'],
    'test_retrieval': ret, 'history': history,
}
with open(ARTIFACT_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('Saved to', ARTIFACT_DIR)

Saved to experiments/siamese_bilstm_artifacts
